# 03 · Baselines, models, and validation

**Objective:** Determine whether a model of tomorrow's QQQ close-to-close
return improves over simple constant forecasts.

**Input:** `data/qqq_model_data.csv`, produced by notebook 02.

**Selection rule:** Lowest **validation RMSE**; also report MAE and the
train–validation gap. No test-set predictions are computed here.


## 1. Load and split chronologically

We allocate ~70% training, ~15% validation, and ~15% final test, without
shuffling. We skip the last row of the training block and the last row
of the validation block. Because each label uses the **next day's**
return, these one-row gaps stop a label from reaching into the next
partition. Future features or targets must not influence fitting.


In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import LinearRegression

project_dir = Path.cwd()
if project_dir.name == "notebooks":
    project_dir = project_dir.parent
sys.path.insert(0, str(project_dir))

from src.qqq_utils import (
    FEATURE_COLUMNS, TARGET_COLUMN, SELECTED_RF_PARAMS,
    chronological_split, load_model_data, regression_metrics,
)

model_data = load_model_data(project_dir)
train_data, val_data, test_data = chronological_split(model_data)

for name, block in [("Training", train_data),
                    ("Validation", val_data), ("Testing (reserved)", test_data)]:
    print(f"{name}: {len(block)} rows, {block.index.min().date()}"
          f" to {block.index.max().date()}")


Training: 2918 rows, 2010-02-02 to 2021-09-02
Validation: 624 rows, 2021-09-07 to 2024-02-29
Testing (reserved): 626 rows, 2024-03-04 to 2026-08-31


In [2]:
X_train, y_train = train_data[FEATURE_COLUMNS], train_data[TARGET_COLUMN]
X_val, y_val = val_data[FEATURE_COLUMNS], val_data[TARGET_COLUMN]

# These dictionaries collect results ONCE for the final comparison.
train_predictions = {}
val_predictions = {}


## 2. Baselines — establish what improvement means

A model is useful only relative to a reference. Our two baselines
always predict either zero return or the **training-period** average
return. We never estimate a baseline using validation or test labels.


In [3]:
training_mean = y_train.mean()
train_predictions["Predict 0%"] = np.zeros(len(y_train))
val_predictions["Predict 0%"] = np.zeros(len(y_val))
train_predictions["Predict training mean"] = np.full(len(y_train), training_mean)
val_predictions["Predict training mean"] = np.full(len(y_val), training_mean)
print(f"Training-only mean daily return: {training_mean:.6f}%")


Training-only mean daily return: 0.085826%


## 3. Linear regression — a simple fitted benchmark

Ordinary least squares fits a linear relationship between four
same-day indicators and the *next-day* return. Coefficients reflect
association, not causality.


In [4]:
linear_model = LinearRegression().fit(X_train, y_train)
train_predictions["Linear regression"] = linear_model.predict(X_train)
val_predictions["Linear regression"] = linear_model.predict(X_val)

linear_coefficients = pd.DataFrame({
    "Feature": FEATURE_COLUMNS, "Coefficient": linear_model.coef_,
})
print("Intercept:", linear_model.intercept_)
linear_coefficients


Intercept: 0.023586094403337733


,Feature,Coefficient
0,Daily_Return_Pct,-0.116343
1,Momentum_5d_Pct,-0.000130
2,Volatility_20d_Pct,0.072380
3,MA_Distance_20d_Pct,-0.007839


## 4. Random Forest — nonlinear relationships

An ensemble of trees can model thresholds and interactions that OLS
cannot. We start with a regularized forest instead of fully grown
trees, aiming to reduce overfitting on volatile financial returns.


In [5]:
rf_model = RandomForestRegressor(
    n_estimators=200, max_depth=5, min_samples_leaf=20,
    random_state=42, n_jobs=-1,
).fit(X_train, y_train)
train_predictions["Random Forest"] = rf_model.predict(X_train)
val_predictions["Random Forest"] = rf_model.predict(X_val)


## 5. Tune only on the validation period

We compare nine predefined `(max_depth, min_samples_leaf)` settings.
Smaller depth and larger leaves limit how precisely trees can fit
noise. We choose the lowest validation RMSE, with the caution that
repeated validation comparisons can themselves overfit this period.


In [6]:
tuning_results = []
for depth in [3, 5, 8]:
    for leaf in [10, 20, 30]:
        candidate = RandomForestRegressor(
            n_estimators=200, max_depth=depth,
            min_samples_leaf=leaf, random_state=42, n_jobs=-1,
        ).fit(X_train, y_train)
        tuning_results.append({
            "Max Depth": depth,
            "Min Samples Leaf": leaf,
            "Train RMSE": regression_metrics(
                y_train, candidate.predict(X_train))["RMSE"],
            "Val RMSE": regression_metrics(
                y_val, candidate.predict(X_val))["RMSE"],
        })

tuning_comparison = pd.DataFrame(tuning_results).sort_values(
    "Val RMSE", ascending=True
).reset_index(drop=True)
tuning_comparison


,Max Depth,Min Samples Leaf,Train RMSE,Val RMSE
0,3,10,1.198728,1.551357
1,3,20,1.209515,1.553724
2,5,10,1.185630,1.555395
3,5,20,1.200251,1.556383
4,8,20,1.182568,1.558756
5,5,30,1.205073,1.559715
6,3,30,1.213196,1.560893
7,8,10,1.164134,1.561163
8,8,30,1.189884,1.563357


In [7]:
best_rf_params = tuning_comparison.iloc[0]
best_depth = int(best_rf_params["Max Depth"])
best_leaf = int(best_rf_params["Min Samples Leaf"])

# Guard against accidentally changing the frozen selection in notebooks 04/05.
assert best_depth == SELECTED_RF_PARAMS["max_depth"]
assert best_leaf == SELECTED_RF_PARAMS["min_samples_leaf"]

tuned_rf_model = RandomForestRegressor(
    n_estimators=200, max_depth=best_depth,
    min_samples_leaf=best_leaf, random_state=42, n_jobs=-1,
).fit(X_train, y_train)
train_predictions["Tuned Random Forest"] = tuned_rf_model.predict(X_train)
val_predictions["Tuned Random Forest"] = tuned_rf_model.predict(X_val)
print(f"Selected: max_depth={best_depth}, min_samples_leaf={best_leaf}")


Selected: max_depth=3, min_samples_leaf=10


## 6. Gradient Boosting — another nonlinear comparison

Sequentially fitted shallow trees learn residual patterns. The
configuration below is a fixed candidate, evaluated on the same
training and validation splits—not the held-out test data.


In [8]:
gb_model = GradientBoostingRegressor(
    n_estimators=200, learning_rate=0.05, max_depth=2,
    min_samples_leaf=20, loss="squared_error", random_state=42,
).fit(X_train, y_train)
train_predictions["Gradient Boosting"] = gb_model.predict(X_train)
val_predictions["Gradient Boosting"] = gb_model.predict(X_val)


## 7. Compare every model once

**RMSE** is our selection metric: squaring penalizes large forecast
misses more strongly. **MAE** is a useful secondary measure of typical
absolute error. Both are in *percentage points*, not dollars. The
train-versus-validation difference helps diagnose overfitting.


In [9]:
rows = []
for name, train_pred in train_predictions.items():
    train_errors = regression_metrics(y_train, train_pred)
    val_errors = regression_metrics(y_val, val_predictions[name])
    rows.append({"Model": name,
                 "Train MAE": train_errors["MAE"],
                 "Train RMSE": train_errors["RMSE"],
                 "Val MAE": val_errors["MAE"],
                 "Val RMSE": val_errors["RMSE"]})

performance_comparison = pd.DataFrame(rows).sort_values(
    "Val RMSE"
).reset_index(drop=True)
performance_comparison.round(6)


,Model,Train MAE,Train RMSE,Val MAE,Val RMSE
0,Tuned Random Forest,0.823029,1.198728,1.182947,1.551357
1,Predict 0%,0.839313,1.236036,1.179289,1.551776
2,Predict training mean,0.831797,1.233053,1.178920,1.551973
3,Random Forest,0.819588,1.200251,1.185997,1.556383
4,Gradient Boosting,0.806981,1.160559,1.189101,1.562895
5,Linear regression,0.833122,1.222699,1.204841,1.566878


## 8. Leakage audit and model freeze

- Dates are sorted and unique; no random train/test shuffle occurred.
- Feature construction uses date t or earlier; the target is t+1.
- Boundary rows are omitted so next-day labels stay within partitions.
- Fitting and the training-mean baseline use only training labels.
- Hyperparameter selection used validation RMSE, never the test set.

**Recorded validation result:** tuned Random Forest RMSE ≈ 1.551357,
zero baseline ≈ 1.551777, training-mean baseline ≈ 1.551974. That
difference is tiny. We therefore freeze the selected forest
(`200` trees, depth `3`, minimum leaf `10`, seed `42`) and reserve
the final test evaluation for notebook 04.

The investigation has *not* established statistically significant
predictive value or a profitable trading strategy.
